# Sci/Tech News Q&A with Gemma (RAG) + Visual Analytics
**Track:** Build w Gemma — Gemma Challenge (Google Developer Group) · Hack Fiesta Miami, Aug 2026

This notebook implements a retrieval-augmented generation (RAG) pipeline on the **20 Newsgroups** dataset (`SetFit/20_newsgroups`), plus a scikit-learn analysis layer and five designed charts:

1. Load 20 Newsgroups from Hugging Face and keep only the **Sci/Tech** groups (`sci.*` and `comp.*`, 9 subtopics).
2. Clean, chunk and embed the posts with a small sentence-embedding model (metadata is kept per chunk).
3. Build a **FAISS** similarity index, with optional metadata filters (subtopic / group).
4. Use **scikit-learn** to measure how well the embedding space separates the subtopics (KMeans, Logistic Regression, PCA).
5. Draw a **Marimekko**, **dendrogram**, **stream graph**, **radial bar charts** and a **3D PCA volume plot** in one high-contrast palette with extruded (3D-look) shapes.
6. Feed retrieved context to Gemma (`google/gemma-2-2b-it`) and generate a cited answer.
7. Export the curated corpus, embeddings and FAISS index.

## 1. Install dependencies

In [1]:
!pip install -qq --only-binary=:all: transformers accelerate datasets sentence-transformers faiss-cpu bitsandbytes
!pip install -qq --only-binary=:all: pandas==2.2.2 matplotlib==3.8.4 scikit-learn==1.3.2 scipy==1.11.4 numpy==1.26.4

print("Environment dependencies normalized.")

ERROR: Ignored the following yanked versions: 3.0.4
ERROR: Could not find a version that satisfies the requirement pandas==2.2.2 (from versions: 2.2.3, 2.3.0, 2.3.1, 2.3.2, 2.3.3, 3.0.0rc0, 3.0.0rc1, 3.0.0rc2, 3.0.0, 3.0.1, 3.0.2, 3.0.3, 3.0.5, 3.0.6)
ERROR: No matching distribution found for pandas==2.2.2
Environment dependencies normalized.


## 2. Configuration

In [2]:
import re, json
import numpy as np
import pandas as pd
from datasets import load_dataset

SEED = 42
KEEP_PREFIXES = ("sci.", "comp.")
MAX_TRAIN_PER_TOPIC = 300
MAX_TEST_PER_TOPIC = 100
MIN_WORDS = 30
CHUNK_WORDS, CHUNK_OVERLAP, MAX_CHUNKS_PER_DOC = 120, 20, 4
EMBED_MODEL = "all-MiniLM-L6-v2"
GEMMA_MODEL_ID = "google/gemma-2-2b-it"

ds = load_dataset("SetFit/20_newsgroups")
print(ds)

NEWSGROUPS = ["comp.graphics", "comp.sys.ibm.pc.hardware",
              "comp.sys.mac.hardware", "comp.windows.x", "misc.forsale", "sci.crypt", "sci.electronics", "sci.med",
              "sci.space"]

def to_df(split):
    df = split.to_pandas()
    if "label_text" not in df.columns:
        df["label_text"] = df["label"].map(lambda i: NEWSGROUPS[i])
    return df

raw_train, raw_test = to_df(ds["train"]), to_df(ds["test"])
print("Raw rows:", len(raw_train), "train /", len(raw_test), "test")
print(raw_train.label_text.value_counts().head(20))

README.md:   0%|          | 0.00/734 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.jsonl: reconstructing file:   0%|          |  0.00B / 14.8MB            

train.jsonl: downloading bytes:           |  0.00B            

test.jsonl: reconstructing file:   0%|          |  0.00B / 8.91MB            

test.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/11314 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7532 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 11314
    })
    test: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 7532
    })
})
Raw rows: 11314 train / 7532 test
label_text
rec.sport.hockey            600
soc.religion.christian      599
rec.motorcycles             598
rec.sport.baseball          597
sci.crypt                   595
sci.med                     594
rec.autos                   594
sci.space                   593
comp.windows.x              593
comp.os.ms-windows.misc     591
sci.electronics             591
comp.sys.ibm.pc.hardware    590
misc.forsale                585
comp.graphics               584
comp.sys.mac.hardware       578
talk.politics.mideast       564
talk.politics.guns          546
alt.atheism                 480
talk.politics.misc          465
talk.religion.misc          377
Name: count, dtype: int64
